# Cefiderocol FAERS Signal Detection Analysis
**Pharmacovigilance Data Science Portfolio**  
**Author:** Michael Olszewski, PharmD, BCPS, BCCCP  
**Data Source:** OpenFDA FAERS API  
**Methodology:** Proportional Reporting Ratio (PRR) — Evans' Criteria  
**Date:** 2026-05-05

---


## 1. Background

Cefiderocol is a siderophore cephalosporin active against carbapenem-resistant 
Gram-negative organisms including *Pseudomonas aeruginosa*, *Acinetobacter baumannii*, 
and carbapenem-resistant Enterobacteriaceae. Given its last-resort status in critically 
ill patients, robust post-marketing pharmacovigilance is essential.

This analysis demonstrates a reproducible signal detection pipeline using the OpenFDA 
FAERS API, implementing PRR disproportionality analysis against a beta-lactam reference 
population (meropenem, n=5,000).

**Regulatory context:** ICH E2C(R2), ICH E2E — signal detection methodology aligned 
with Evans et al. (2001) classical threshold criteria: PRR ≥ 2, N ≥ 3, χ² ≥ 4.


## 2. Data Acquisition

In [ ]:
import requests
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
from pathlib import Path
from datetime import datetime

# OpenFDA API configuration
API_BASE = "https://api.fda.gov/drug/event.json"

def fetch_reports(drug_name, max_records=None, limit=100):
    reports, skip = [], 0
    while True:
        if max_records and len(reports) >= max_records:
            break
        params = {"search": f"patient.drug.medicinalproduct:{drug_name}",
                  "limit": limit, "skip": skip}
        r = requests.get(API_BASE, params=params, timeout=15)
        if r.status_code != 200:
            break
        batch = r.json().get("results", [])
        if not batch:
            break
        reports.extend(batch)
        if len(batch) < limit:
            break
        skip += limit
        time.sleep(0.25)
    return reports

print("Loading pre-fetched signal data from pipeline run...")
print(f"Data source: OpenFDA FAERS API")
print(f"Drug: Cefiderocol | Reference: Meropenem (n=5,000)")
print(f"Pipeline completed: 2026-05-05")


## 3. PRR Signal Detection Results

In [ ]:
# Load pipeline results
signals = [
  {
    "reaction_pt": "no adverse event",
    "drug_cases": 25,
    "prr": 28.755,
    "chi2": 147.35,
    "signal": True
  },
  {
    "reaction_pt": "parainfluenzae virus infection",
    "drug_cases": 7,
    "prr": 28.18,
    "chi2": 40.85,
    "signal": True
  },
  {
    "reaction_pt": "therapeutic product effect incomplete",
    "drug_cases": 3,
    "prr": 24.155,
    "chi2": 16.66,
    "signal": True
  },
  {
    "reaction_pt": "complications of transplanted lung",
    "drug_cases": 3,
    "prr": 24.155,
    "chi2": 16.66,
    "signal": True
  },
  {
    "reaction_pt": "ischaemic hepatitis",
    "drug_cases": 3,
    "prr": 24.155,
    "chi2": 16.66,
    "signal": True
  },
  {
    "reaction_pt": "therapy non-responder",
    "drug_cases": 11,
    "prr": 17.713,
    "chi2": 54.36,
    "signal": True
  },
  {
    "reaction_pt": "pneumonia pseudomonal",
    "drug_cases": 8,
    "prr": 16.103,
    "chi2": 37.85,
    "signal": True
  },
  {
    "reaction_pt": "treatment failure",
    "drug_cases": 54,
    "prr": 9.881,
    "chi2": 196.97,
    "signal": True
  },
  {
    "reaction_pt": "rhinorrhoea",
    "drug_cases": 7,
    "prr": 9.393,
    "chi2": 24.29,
    "signal": True
  },
  {
    "reaction_pt": "death",
    "drug_cases": 109,
    "prr": 9.336,
    "chi2": 389.77,
    "signal": True
  },
  {
    "reaction_pt": "abdominal infection",
    "drug_cases": 3,
    "prr": 8.052,
    "chi2": 9.27,
    "signal": True
  },
  {
    "reaction_pt": "ototoxicity",
    "drug_cases": 9,
    "prr": 7.246,
    "chi2": 25.59,
    "signal": True
  },
  {
    "reaction_pt": "lipase increased",
    "drug_cases": 5,
    "prr": 6.71,
    "chi2": 13.28,
    "signal": True
  },
  {
    "reaction_pt": "intestinal ischaemia",
    "drug_cases": 3,
    "prr": 6.039,
    "chi2": 7.22,
    "signal": True
  },
  {
    "reaction_pt": "enterococcal sepsis",
    "drug_cases": 3,
    "prr": 6.039,
    "chi2": 7.22,
    "signal": True
  },
  {
    "reaction_pt": "drug ineffective for unapproved indication",
    "drug_cases": 12,
    "prr": 5.085,
    "chi2": 24.27,
    "signal": True
  },
  {
    "reaction_pt": "nephropathy toxic",
    "drug_cases": 9,
    "prr": 4.529,
    "chi2": 15.91,
    "signal": True
  },
  {
    "reaction_pt": "encephalitis",
    "drug_cases": 6,
    "prr": 4.026,
    "chi2": 9.13,
    "signal": True
  },
  {
    "reaction_pt": "lymphoma",
    "drug_cases": 3,
    "prr": 3.451,
    "chi2": 3.66,
    "signal": False
  },
  {
    "reaction_pt": "myoclonus",
    "drug_cases": 5,
    "prr": 3.097,
    "chi2": 5.14,
    "signal": True
  },
  {
    "reaction_pt": "lung abscess",
    "drug_cases": 3,
    "prr": 3.019,
    "chi2": 2.95,
    "signal": False
  },
  {
    "reaction_pt": "haemolytic anaemia",
    "drug_cases": 4,
    "prr": 2.928,
    "chi2": 3.73,
    "signal": False
  },
  {
    "reaction_pt": "pneumonitis",
    "drug_cases": 8,
    "prr": 2.684,
    "chi2": 6.37,
    "signal": True
  },
  {
    "reaction_pt": "drug resistance",
    "drug_cases": 60,
    "prr": 2.64,
    "chi2": 48.11,
    "signal": True
  },
  {
    "reaction_pt": "pathogen resistance",
    "drug_cases": 31,
    "prr": 2.573,
    "chi2": 23.12,
    "signal": True
  },
  {
    "reaction_pt": "bradycardia",
    "drug_cases": 6,
    "prr": 2.543,
    "chi2": 4.29,
    "signal": True
  },
  {
    "reaction_pt": "influenza",
    "drug_cases": 4,
    "prr": 2.477,
    "chi2": 2.7,
    "signal": False
  },
  {
    "reaction_pt": "pseudomonas infection",
    "drug_cases": 15,
    "prr": 2.415,
    "chi2": 9.68,
    "signal": True
  },
  {
    "reaction_pt": "acute myeloid leukaemia",
    "drug_cases": 5,
    "prr": 2.368,
    "chi2": 3.07,
    "signal": False
  },
  {
    "reaction_pt": "off label use",
    "drug_cases": 60,
    "prr": 2.357,
    "chi2": 38.04,
    "signal": True
  },
  {
    "reaction_pt": "pancreatitis acute",
    "drug_cases": 5,
    "prr": 2.237,
    "chi2": 2.69,
    "signal": False
  },
  {
    "reaction_pt": "generalised tonic-clonic seizure",
    "drug_cases": 3,
    "prr": 2.196,
    "chi2": 1.54,
    "signal": False
  },
  {
    "reaction_pt": "coagulopathy",
    "drug_cases": 10,
    "prr": 2.119,
    "chi2": 4.72,
    "signal": True
  },
  {
    "reaction_pt": "chromaturia",
    "drug_cases": 4,
    "prr": 1.894,
    "chi2": 1.37,
    "signal": False
  },
  {
    "reaction_pt": "cholangitis",
    "drug_cases": 4,
    "prr": 1.789,
    "chi2": 1.14,
    "signal": False
  },
  {
    "reaction_pt": "bacteraemia",
    "drug_cases": 11,
    "prr": 1.703,
    "chi2": 2.67,
    "signal": False
  },
  {
    "reaction_pt": "posterior reversible encephalopathy syndrome",
    "drug_cases": 4,
    "prr": 1.695,
    "chi2": 0.95,
    "signal": False
  },
  {
    "reaction_pt": "disease recurrence",
    "drug_cases": 4,
    "prr": 1.695,
    "chi2": 0.95,
    "signal": False
  },
  {
    "reaction_pt": "stenotrophomonas infection",
    "drug_cases": 4,
    "prr": 1.61,
    "chi2": 0.77,
    "signal": False
  },
  {
    "reaction_pt": "liver injury",
    "drug_cases": 3,
    "prr": 1.61,
    "chi2": 0.58,
    "signal": False
  },
  {
    "reaction_pt": "fungaemia",
    "drug_cases": 4,
    "prr": 1.61,
    "chi2": 0.77,
    "signal": False
  },
  {
    "reaction_pt": "erythema",
    "drug_cases": 8,
    "prr": 1.571,
    "chi2": 1.4,
    "signal": False
  },
  {
    "reaction_pt": "cough",
    "drug_cases": 9,
    "prr": 1.542,
    "chi2": 1.45,
    "signal": False
  },
  {
    "reaction_pt": "cellulitis",
    "drug_cases": 7,
    "prr": 1.523,
    "chi2": 1.07,
    "signal": False
  },
  {
    "reaction_pt": "myelodysplastic syndrome",
    "drug_cases": 3,
    "prr": 1.51,
    "chi2": 0.44,
    "signal": False
  },
  {
    "reaction_pt": "abscess",
    "drug_cases": 3,
    "prr": 1.51,
    "chi2": 0.44,
    "signal": False
  },
  {
    "reaction_pt": "drug ineffective",
    "drug_cases": 87,
    "prr": 1.494,
    "chi2": 13.28,
    "signal": False
  },
  {
    "reaction_pt": "systemic candida",
    "drug_cases": 5,
    "prr": 1.491,
    "chi2": 0.69,
    "signal": False
  },
  {
    "reaction_pt": "encephalopathy",
    "drug_cases": 8,
    "prr": 1.431,
    "chi2": 0.89,
    "signal": False
  },
  {
    "reaction_pt": "multiple organ dysfunction syndrome",
    "drug_cases": 28,
    "prr": 1.409,
    "chi2": 2.93,
    "signal": False
  },
  {
    "reaction_pt": "hepatic enzyme increased",
    "drug_cases": 7,
    "prr": 1.375,
    "chi2": 0.62,
    "signal": False
  },
  {
    "reaction_pt": "leukocytosis",
    "drug_cases": 4,
    "prr": 1.342,
    "chi2": 0.3,
    "signal": False
  },
  {
    "reaction_pt": "toxicity to various agents",
    "drug_cases": 9,
    "prr": 1.208,
    "chi2": 0.28,
    "signal": False
  },
  {
    "reaction_pt": "hepatotoxicity",
    "drug_cases": 4,
    "prr": 1.193,
    "chi2": 0.11,
    "signal": False
  },
  {
    "reaction_pt": "product use in unapproved indication",
    "drug_cases": 8,
    "prr": 1.171,
    "chi2": 0.18,
    "signal": False
  },
  {
    "reaction_pt": "septic shock",
    "drug_cases": 29,
    "prr": 1.081,
    "chi2": 0.16,
    "signal": False
  },
  {
    "reaction_pt": "renal impairment",
    "drug_cases": 15,
    "prr": 1.05,
    "chi2": 0.03,
    "signal": False
  },
  {
    "reaction_pt": "peritonitis",
    "drug_cases": 4,
    "prr": 1.039,
    "chi2": 0.01,
    "signal": False
  },
  {
    "reaction_pt": "renal failure",
    "drug_cases": 17,
    "prr": 1.021,
    "chi2": 0.01,
    "signal": False
  },
  {
    "reaction_pt": "blood alkaline phosphatase increased",
    "drug_cases": 4,
    "prr": 1.006,
    "chi2": 0.0,
    "signal": False
  },
  {
    "reaction_pt": "cholestasis",
    "drug_cases": 5,
    "prr": 1.006,
    "chi2": 0.0,
    "signal": False
  },
  {
    "reaction_pt": "disease progression",
    "drug_cases": 5,
    "prr": 1.006,
    "chi2": 0.0,
    "signal": False
  },
  {
    "reaction_pt": "delirium",
    "drug_cases": 5,
    "prr": 1.006,
    "chi2": 0.0,
    "signal": False
  },
  {
    "reaction_pt": "toxic epidermal necrolysis",
    "drug_cases": 12,
    "prr": 0.996,
    "chi2": 0.0,
    "signal": False
  },
  {
    "reaction_pt": "acute respiratory failure",
    "drug_cases": 4,
    "prr": 0.976,
    "chi2": 0.0,
    "signal": False
  },
  {
    "reaction_pt": "pneumonia",
    "drug_cases": 32,
    "prr": 0.969,
    "chi2": 0.03,
    "signal": False
  },
  {
    "reaction_pt": "acute kidney injury",
    "drug_cases": 27,
    "prr": 0.962,
    "chi2": 0.04,
    "signal": False
  },
  {
    "reaction_pt": "thrombocytopenia",
    "drug_cases": 18,
    "prr": 0.878,
    "chi2": 0.28,
    "signal": False
  },
  {
    "reaction_pt": "sepsis",
    "drug_cases": 28,
    "prr": 0.867,
    "chi2": 0.54,
    "signal": False
  },
  {
    "reaction_pt": "neutropenia",
    "drug_cases": 14,
    "prr": 0.86,
    "chi2": 0.29,
    "signal": False
  },
  {
    "reaction_pt": "disseminated intravascular coagulation",
    "drug_cases": 8,
    "prr": 0.859,
    "chi2": 0.17,
    "signal": False
  },
  {
    "reaction_pt": "condition aggravated",
    "drug_cases": 12,
    "prr": 0.855,
    "chi2": 0.27,
    "signal": False
  },
  {
    "reaction_pt": "hyperkalaemia",
    "drug_cases": 3,
    "prr": 0.805,
    "chi2": 0.13,
    "signal": False
  },
  {
    "reaction_pt": "fungal infection",
    "drug_cases": 3,
    "prr": 0.779,
    "chi2": 0.17,
    "signal": False
  },
  {
    "reaction_pt": "infection",
    "drug_cases": 7,
    "prr": 0.772,
    "chi2": 0.44,
    "signal": False
  },
  {
    "reaction_pt": "hypertension",
    "drug_cases": 3,
    "prr": 0.69,
    "chi2": 0.39,
    "signal": False
  },
  {
    "reaction_pt": "pulmonary embolism",
    "drug_cases": 3,
    "prr": 0.69,
    "chi2": 0.39,
    "signal": False
  },
  {
    "reaction_pt": "general physical health deterioration",
    "drug_cases": 6,
    "prr": 0.68,
    "chi2": 0.84,
    "signal": False
  },
  {
    "reaction_pt": "hepatic function abnormal",
    "drug_cases": 7,
    "prr": 0.633,
    "chi2": 1.4,
    "signal": False
  },
  {
    "reaction_pt": "acute respiratory distress syndrome",
    "drug_cases": 4,
    "prr": 0.596,
    "chi2": 1.03,
    "signal": False
  },
  {
    "reaction_pt": "febrile neutropenia",
    "drug_cases": 13,
    "prr": 0.591,
    "chi2": 3.54,
    "signal": False
  },
  {
    "reaction_pt": "pneumonia aspiration",
    "drug_cases": 3,
    "prr": 0.525,
    "chi2": 1.22,
    "signal": False
  },
  {
    "reaction_pt": "epilepsy",
    "drug_cases": 3,
    "prr": 0.503,
    "chi2": 1.4,
    "signal": False
  },
  {
    "reaction_pt": "aspartate aminotransferase increased",
    "drug_cases": 4,
    "prr": 0.488,
    "chi2": 2.05,
    "signal": False
  },
  {
    "reaction_pt": "drug reaction with eosinophilia and systemic symptoms",
    "drug_cases": 8,
    "prr": 0.421,
    "chi2": 6.23,
    "signal": False
  },
  {
    "reaction_pt": "white blood cell count decreased",
    "drug_cases": 4,
    "prr": 0.418,
    "chi2": 3.12,
    "signal": False
  },
  {
    "reaction_pt": "hypersensitivity",
    "drug_cases": 3,
    "prr": 0.416,
    "chi2": 2.36,
    "signal": False
  },
  {
    "reaction_pt": "rash",
    "drug_cases": 8,
    "prr": 0.416,
    "chi2": 6.44,
    "signal": False
  },
  {
    "reaction_pt": "respiratory failure",
    "drug_cases": 8,
    "prr": 0.372,
    "chi2": 8.36,
    "signal": False
  },
  {
    "reaction_pt": "alanine aminotransferase increased",
    "drug_cases": 4,
    "prr": 0.366,
    "chi2": 4.27,
    "signal": False
  },
  {
    "reaction_pt": "nausea",
    "drug_cases": 5,
    "prr": 0.282,
    "chi2": 9.1,
    "signal": False
  },
  {
    "reaction_pt": "leukopenia",
    "drug_cases": 3,
    "prr": 0.274,
    "chi2": 5.65,
    "signal": False
  },
  {
    "reaction_pt": "anaemia",
    "drug_cases": 5,
    "prr": 0.26,
    "chi2": 10.52,
    "signal": False
  },
  {
    "reaction_pt": "diarrhoea",
    "drug_cases": 5,
    "prr": 0.241,
    "chi2": 11.97,
    "signal": False
  },
  {
    "reaction_pt": "hypotension",
    "drug_cases": 3,
    "prr": 0.159,
    "chi2": 13.47,
    "signal": False
  },
  {
    "reaction_pt": "pyrexia",
    "drug_cases": 7,
    "prr": 0.131,
    "chi2": 43.16,
    "signal": False
  }
]

df = pd.DataFrame(signals)
df_signals = df[df['signal'] == True].copy()
df_all = df.copy()

print(f"Total reaction PTs analyzed: {len(df)}")
print(f"Signals detected (PRR≥2, N≥3, χ²≥4): {len(df_signals)}")
print()
print("=" * 70)
print(f"{' Cefiderocol FAERS Signal Detection — Top 20 by PRR ':=^70}")
print("=" * 70)
display(df_signals[['reaction_pt','drug_cases','prr','chi2']].head(20).rename(columns={
    'reaction_pt': 'MedDRA PT',
    'drug_cases': 'N (cases)',
    'prr': 'PRR',
    'chi2': 'Chi²'
}).reset_index(drop=True))


## 4. Visualization

In [ ]:
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Cefiderocol FAERS Disproportionality Analysis\nPRR Signal Detection vs Meropenem Reference Population',
             fontsize=13, fontweight='bold', y=1.02)

# Plot 1: Top 15 signals by PRR
top15 = df_signals.nlargest(15, 'prr')
# Filter out data quality artifact
top15 = top15[top15['reaction_pt'] != 'no adverse event']

colors = ['#c0392b' if p >= 10 else '#e67e22' if p >= 5 else '#f1c40f' for p in top15['prr']]
bars = axes[0].barh(top15['reaction_pt'], top15['prr'], color=colors)
axes[0].axvline(x=2, color='black', linestyle='--', alpha=0.5, label='PRR threshold (2.0)')
axes[0].set_xlabel('Proportional Reporting Ratio (PRR)', fontsize=11)
axes[0].set_title('Top Signals by PRR Magnitude', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].invert_yaxis()

# Add value labels
for bar, val in zip(bars, top15['prr']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=8)

# Plot 2: PRR vs N bubble chart
clinical = df_signals[df_signals['reaction_pt'] != 'no adverse event']
scatter = axes[1].scatter(clinical['drug_cases'], clinical['prr'],
                          s=clinical['chi2']*3, alpha=0.6,
                          c=clinical['prr'], cmap='YlOrRd')
axes[1].axhline(y=2, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Number of Cases (N)', fontsize=11)
axes[1].set_ylabel('PRR', fontsize=11)
axes[1].set_title('PRR vs Case Count\n(bubble size = χ² magnitude)', fontsize=11, fontweight='bold')
plt.colorbar(scatter, ax=axes[1], label='PRR')

# Label key points
for _, row in clinical[clinical['drug_cases'] > 10].iterrows():
    axes[1].annotate(row['reaction_pt'],
                    (row['drug_cases'], row['prr']),
                    fontsize=7, xytext=(5, 5),
                    textcoords='offset points')

plt.tight_layout()
plt.savefig(Path.home() / 'pv_workbench/output/cefiderocol_prr_chart.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")


## 5. Clinical Interpretation

### Key Signals — ICU Pharmacist Perspective

| MedDRA PT | N | PRR | Clinical Assessment |
|---|---|---|---|
| Pneumonia pseudomonal | 8 | 16.10 | **Expected** — cefiderocol's primary indication; confirms on-label use in resistant pseudomonal pneumonia |
| Treatment failure | 54 | 9.88 | **Clinically significant** — aligns with published RCTs showing non-inferiority concerns vs carbapenems in some subgroups |
| Death | 109 | 9.34 | **Confounding by indication** — reflects critically ill, last-resort patient population; not a de novo safety signal |
| Therapy non-responder | 11 | 17.71 | **Warrants monitoring** — may reflect emerging resistance; correlates with MIC creep literature |
| Ischaemic hepatitis | 3 | 24.16 | **Hypothesis-generating** — low N but high PRR; requires further evaluation against complete FAERS database |
| Complications of transplanted lung | 3 | 24.16 | **Population signal** — lung transplant recipients are a core cefiderocol use case; not unexpected |

### Methodological Limitations

1. **Confounding by indication**: Cefiderocol patients are inherently sicker than the meropenem reference population — high PRR for mortality-related PTs is expected and does not imply causality
2. **OpenFDA subset**: The API exposes a subset of total FAERS reports; a regulatory-grade analysis would use the complete quarterly ASCII files
3. **Reference population**: Meropenem was selected as a clinically relevant comparator; PRR values would differ with a broader background population
4. **Underreporting bias**: Standard FAERS limitation — actual event rates are higher than reported
5. **MedDRA coding variability**: Reporter coding practices affect PT-level analysis


## 6. Regulatory & Technical Context

### Pipeline Architecture
- **Data source**: OpenFDA FAERS API (REST/JSON)
- **Processing**: Python 3.12, containerized via Docker on Debian Linux
- **Signal method**: PRR with Evans' criteria (ICH E2C-aligned)
- **Reference**: Meropenem comparator population (n=5,000)
- **Output**: JSON signal file + formatted report

### Applicability to Vault Safety / Argus Workflows
This pipeline demonstrates the signal detection logic that underlies aggregate 
analysis modules in systems like Veeva Vault Safety and Oracle Argus. The PRR/ROR 
methodology implemented here is consistent with ICH E2E signal management guidance 
and maps directly to the quantitative signal detection workflows used in PSUR/PBRER 
preparation.
